# Proyecto CRUZBER — Conclusiones Finales
## Predicción de Demanda Semanal por SKU y Municipio
### 7 Iteraciones · 2022–2024 · Resumen Ejecutivo Global

---

> **¿Qué problema resolvemos?**
> CRUZBER necesita saber cuántas unidades de cada producto va a vender en cada municipio la semana siguiente. Acertar en esta predicción permite comprar bien, evitar roturas de stock y no inmovilizar capital en exceso de inventario.
>
> Este documento resume el trabajo realizado a lo largo de **7 iteraciones de modelado**, explicando qué se hizo en cada una, qué aprendimos y cuál es el estado actual del modelo.

---

### El punto de partida: ¿por qué es difícil este problema?

Predecir la demanda de artículos deportivos a nivel de municipio y SKU es especialmente complejo por tres razones:

1. **Altísima granularidad**: no predicemos "cuántos SKUs vende CRUZBER esta semana", sino cuántas unidades de cada referencia concreta se venden en cada uno de los miles de municipios donde opera. Son cientos de miles de combinaciones distintas.

2. **Demanda muy irregular**: la mayoría de los productos (Tipo B&C) tienen demanda esporádica — semanas en cero y semanas con pico. No siguen una curva suave.

3. **Múltiples factores**: la venta depende del clima, la temporada ciclista, los descuentos, el canal de venta, el precio... y muchos de estos factores interactúan entre sí.

Dicho esto: el reto es grande, y **cada décima de mejora en el error tiene impacto real en el negocio**.

---

## PARTE 1 — Las 7 Iteraciones: Qué se hizo y por qué

Cada iteración añade una pieza al puzzle. La lógica es siempre la misma: identificar la limitación más importante del modelo anterior, diseñar una solución concreta y medir si el resultado mejora.

---

### Iteración 1 — El punto de partida (Baseline)

**¿Qué se hizo?**
Se construyó el primer modelo funcional usando CatBoost (un algoritmo de machine learning basado en árboles de decisión) con los datos históricos de ventas de 2022-2023 para predecir 2024. Sin ningún tipo de ajuste ni optimización — simplemente el modelo en su configuración por defecto.

**¿Por qué?**
Antes de mejorar algo, hay que saber desde dónde se parte. El modelo baseline es la referencia contra la que mediremos todo lo que venga después. Si no superas al baseline, la mejora propuesta no funciona.

**¿Qué variables usaba?**
Las variables brutas disponibles: semana del año, año, provincia, municipio, código de artículo, familia, subfamilia, canal de venta, temperatura, precipitación, viento, pruebas ciclistas.

**Resultado:**
- Error medio (MAE): **0.793 unidades** por predicción
- Capacidad explicativa (R²): **0.295** — el modelo explica el 29.5% de la variación en la demanda
- La otra mitad del error viene de factores que el modelo todavía no conoce

**En términos de negocio:** si el modelo predice que vas a vender 10 unidades, la realidad está a ±0.8 unidades de media. No es malo para empezar, pero hay margen de mejora.

---

### Iteración 2 — La memoria a corto plazo

**¿Qué se hizo?**
Se añadió una variable nueva: la **media de ventas de las 4 semanas anteriores** (ventana deslizante). Es decir, si las últimas 4 semanas vendiste 8, 10, 9 y 11 unidades, el modelo recibe ese promedio (9.5) como señal.

**¿Por qué?**
El modelo no tenía "memoria". No sabía si ese SKU en ese municipio había estado vendiendo mucho o poco recientemente. La tendencia reciente es uno de los mejores predictores de la demanda futura — si llevas 4 semanas vendiendo bien, probablemente seguirás vendiendo bien la semana que viene.

**Precaución técnica:** para no hacer trampa, la media se calcula siempre sobre semanas anteriores, nunca sobre la semana que se predice (shift de 1 semana).

**Resultado:**
- MAE: **0.773** (−2.5% vs It1) ✓
- R²: **0.330** (+11.9% relativo vs It1) ✓
- El modelo ya "recuerda" lo que ha pasado recientemente

---

### Iteración 3 — La memoria a largo plazo: estacionalidad

**¿Qué se hizo?**
Se añadió otra variable: **las ventas de esa misma semana el año anterior**. Si la semana 20 de 2023 fue buena, probablemente la semana 20 de 2024 también lo será.

**¿Por qué?**
El negocio de artículos deportivos es muy estacional — hay picos en primavera (temporada ciclista), verano y antes de Navidad. Estos patrones se repiten año a año. Darle al modelo la "foto del año pasado en la misma época" es una señal muy potente.

**Resultado:**
- MAE: **0.769** (−0.5% vs It2) — mejora muy pequeña
- R²: **0.330** — igual que It2
- La estacionalidad interanual no aportó tanto como esperábamos. Probablemente porque 2024 tuvo un comportamiento algo diferente a 2023, y el modelo lo interpreta como ruido.

**Aprendizaje clave:** añadir más variables no siempre mejora el modelo. Si la variable no aporta señal nueva, el modelo la ignora o genera ruido.

---

### Iteración 4 — El gran salto: transformar el problema

**¿Qué se hizo?**
Dos cambios importantes a la vez:

1. **Transformación logarítmica del target**: en lugar de predecir directamente las unidades vendidas (que pueden ir de 0 a 500), el modelo predice el logaritmo de las unidades. Esto "comprime" los valores extremos y hace el problema más manejable.

2. **Validación cruzada temporal**: en lugar de una sola prueba (entreno en 2022-2023, evalúo en 2024), el modelo se evalúa en tres periodos distintos de forma rodante. Esto da una estimación más robusta de si el modelo realmente aprende o simplemente memoriza.

3. **Análisis separado A vs B&C**: se evalúan los productos de alta rotación (A) por separado de los de nicho (B&C), porque tienen comportamientos muy distintos.

**¿Por qué?**
El problema con demandas que van de 0 a 500 es que los errores en los picos (cuando vendes 400 y predices 200) dominan el aprendizaje del modelo, que "se olvida" de los productos pequeños. Al trabajar en escala logarítmica, todos los productos tienen el mismo peso.

**Resultado:**
- MAE: **0.649** (−15.6% vs It3) ← **El mayor salto de todo el proyecto**
- MAPE: **26.3%** — primer cálculo de error porcentual
- El modelo pasó de equivocarse en 0.769 unidades a 0.649 unidades de media

**En términos de negocio:** como si de repente el modelo "viese" mejor los productos pequeños, que son la mayoría del catálogo.

---

### Iteración 5 — Búsqueda del mejor modelo y aprendizaje histórico profundo

**¿Qué se hizo?**
Dos mejoras técnicas sofisticadas:

1. **Optuna (búsqueda automática de hiperparámetros)**: en lugar de configurar el modelo a mano, se lanzaron 30 pruebas automáticas para encontrar la combinación óptima de parámetros internos del modelo.

2. **Target Encoding**: se creó una variable nueva que resume el historial de ventas de cada SKU en cada municipio. En lugar de que el modelo "adivine" que ese producto siempre se vende bien en esa localidad, se le da directamente ese número.

3. **Festivos nacionales**: se marcaron las semanas con festivos, esperando capturar el efecto vacacional en las ventas.

**¿Por qué?**
Tras la mejora de It4, el siguiente cuello de botella era que el modelo no conocía el "perfil histórico" de cada combinación SKU-municipio. Un producto que siempre vende 15 unidades en Bilbao debe predecirse diferente a uno que vende 2.

**Resultado:**
- MAE: **0.641** (−1.2% vs It4) — mejora modesta
- Los festivos nacionales tuvieron importancia casi nula (solo 23 semanas festivas en 3 años)
- El target encoding (historial SKU-municipio) se convirtió en la variable más importante: **15.9% de importancia**

**Aprendizaje:** los festivos nacionales son demasiado escasos para generar señal estadística. Los festivos autonómicos (más numerosos y localizados) podrían ser más útiles.

---

### Iteración 6 — Nuevas fuentes de información: descuentos y geografía

**¿Qué se hizo?**
Se incorporaron dos nuevas fuentes de datos:

1. **Descuentos promocionales**: en los datos originales existía una variable de descuento promocional (`por_descuento2`) que nunca se había usado en el modelo. Se procesó y se añadió en dos formas: el % de descuento activo y un indicador binario (¿hubo descuento esta semana?).

2. **Región geográfica**: se agruparon las 52 provincias en 6 regiones climáticas (Noroeste, Norte, Noreste, Centro, Sur, Canarias) para capturar patrones de comportamiento por clima y geografía.

**¿Por qué?**
El análisis de los datos reveló que las semanas con descuento promocional generan una demanda **4.2 veces mayor** que las semanas sin descuento. Era una señal demasiado potente para ignorar.

**Resultado:**
- MAE: **0.641** (prácticamente igual a It5)
- R²: **0.310** (+7.7% relativo) — mejora en capacidad explicativa
- `valor_descuento_promo`: **3.77% de importancia** — la más relevante de las nuevas variables
- La región geográfica aportó muy poco (0.21%) porque la información ya estaba implícita en provincia/municipio

**Aprendizaje:** los descuentos impactan la demanda, pero como solo el 1.5% de las filas tienen descuento activo, el impacto global es limitado. La región geográfica, aunque lógicamente tiene sentido, no añade información que el modelo no pueda inferir de municipio y provincia.

---

### Iteración 7 — La decisión arquitectónica: dos modelos en lugar de uno

**¿Qué se hizo?**
En lugar de un único modelo para todos los productos, se entrenaron **dos modelos independientes**:
- **Modelo A**: entrenado solo con datos de productos de alta rotación (Tipo A)
- **Modelo B&C**: entrenado solo con datos de productos de nicho (Tipo B&C)

Cada modelo buscó sus propios hiperparámetros óptimos con Optuna (40 pruebas cada uno).

**¿Por qué?**
Las iteraciones anteriores habían revelado que A y B&C tienen naturalezas completamente distintas. Un modelo que intenta predecir los dos a la vez aprende un "compromiso" que no es óptimo para ninguno. Es como tener un único comercial que atiende tanto clientes premium como clientes de precio: al final no sirve bien a ninguno.

**Resultado:**
- MAE global: **0.628** (−2.0% vs It6) ✓
- MAPE: **21.5%** (−16.6% relativo vs It6) ← **Mayor mejora en MAPE de todo el proyecto**
- Overfitting muy reducido: brecha RMSE 33-34% (vs 42% en It6) ✓
- Modelo A: MAE **0.676** (−2.0% vs It6 para A)
- Modelo B&C: MAE **0.521** (−1.8% vs It6 para B&C)
- Los hiperparámetros óptimos de A y B&C son **incompatibles** — ningún modelo global podría ser óptimo para ambos a la vez

---

## PARTE 2 — Comparativa de Métricas: La Evolución del Modelo

### El cuadro de mando de las 7 iteraciones

| Iteración | Novedad principal | MAE | MAPE | R² | Δ MAE acum. |
|---|---|:---:|:---:|:---:|:---:|
| **It1** Baseline | Modelo inicial sin ajuste | 0.793 | — | 0.295 | — |
| **It2** Memoria reciente | Media ventas últimas 4 semanas | 0.773 | — | 0.330 | −2.5% |
| **It3** Memoria anual | Ventas misma semana año anterior | 0.769 | — | 0.330 | −3.0% |
| **It4** Escala log | Transformación logarítmica del target | 0.649 | 26.3% | 0.287 | **−18.2%** |
| **It5** Historial por SKU | Optuna + encoding histórico | 0.641 | 26.0% | 0.288 | −19.2% |
| **It6** Descuentos y región | Variables promocionales + geografía | 0.641 | 25.8% | 0.310 | −19.2% |
| **It7** Dos modelos | Modelos dedicados A y B&C | **0.628** | **21.5%** | 0.264* | **−20.8%** |

*El R² de It7 es más bajo por efecto estadístico de combinar dos poblaciones con medias distintas, no porque el modelo sea peor. Cada modelo individual tiene buen ajuste.

### ¿Cómo leer estas métricas?

- **MAE (Error Absoluto Medio)**: en promedio, ¿cuántas unidades nos equivocamos? Si MAE = 0.628, nos equivocamos en 0.628 unidades por predicción.
- **MAPE (Error Porcentual Medio)**: ¿qué porcentaje del valor real es nuestro error? Si MAPE = 21.5% y la demanda real es 10 unidades, nuestro error es de ≈2.15 unidades.
- **R²**: ¿qué porcentaje de la variabilidad de la demanda explica el modelo? R²=0.31 significa que el 31% de "por qué sube o baja la demanda" lo explica el modelo. El 69% restante depende de factores no disponibles.

### Los dos grandes saltos

**Salto 1 (It3→It4): −15.6% en MAE**
La transformación logarítmica del target. Este es el cambio más impactante de todo el proyecto porque no añadió nueva información — simplemente transformó el problema para que el modelo pudiera aprenderlo mejor.

**Salto 2 (It6→It7): −16.6% en MAPE**
La separación en dos modelos dedicados. Sin añadir nuevos datos, simplemente especializando el modelo para cada tipo de producto, el error porcentual baja más de 4 puntos.

### El techo del modelo global

Entre It4 e It6 (tres iteraciones), el MAE apenas mejoró: de 0.649 a 0.641, solo −1.2%. Esto indica que el **modelo global había llegado a su límite**: con una arquitectura única para todos los productos, no era posible mejorar más añadiendo variables. La solución fue cambiar la arquitectura (It7), no añadir más datos.

---

## PARTE 3 — Las Variables que Más Importan (y por qué)

Una de las ventajas del algoritmo CatBoost es que al final del entrenamiento nos dice qué variables influyeron más en sus predicciones. Esto es información de negocio valiosa más allá de la predicción en sí.

### Evolución de las variables más importantes por iteración

| # | Iteración | Variable #1 | Variable #2 | Variable #3 |
|---|---|---|---|---|
| It1-3 | Baseline + memoria | Semana del año | Provincia | Canal de venta |
| It4 | Log1p | Media 4 semanas | Semana del año | Provincia |
| It5 | Encoding histórico | **Historial SKU-Municipio (15.9%)** | Precio unitario | Media 4 semanas |
| It6 | Descuentos | **Historial SKU-Municipio (19.1%)** | Precio unitario | Media 4 semanas |
| It7-A | Modelo A | **Historial SKU-Municipio (68.6%)** | Media 4 semanas | Descuento promo |
| It7-BC | Modelo B&C | **Historial SKU-Municipio (53.7%)** | Precio unitario | Media 4 semanas |

### ¿Qué nos dice la importancia de variables?

**1. El historial propio del SKU lo es casi todo**
Desde que se introdujo en It5, el historial de ventas de cada referencia en cada municipio (*target encoding*) domina el modelo. En It7, para los productos A llega al 68.6%. Traducción: la mejor señal que tenemos de cuánto se va a vender un SKU en Bilbao la semana que viene es cuánto se ha vendido históricamente en Bilbao.

**2. El precio, clave para B&C**
Para los productos de nicho (B&C), el precio unitario tiene una importancia del 19.2%, frente al 4.6% en productos A. Esto tiene sentido: cuando compras un accesorio de nicho (no un best-seller), el precio influye mucho más en la decisión de compra.

**3. El descuento promocional importa más en A**
Los productos estrella (A) son los que más se benefician de las campañas promocionales. Un descuento en un producto de alta rotación genera más ventas adicionales que en un producto de nicho.

**4. El clima importa... pero menos de lo esperado**
La temperatura y la precipitación tienen relevancia moderada. En los modelos dedicados, el clima solo aparece en B&C (1.4% de importancia). Puede parecer sorprendente, pero tiene lógica: si ya sabes cuánto vendiste esa semana el año pasado (que también tenía su clima), el clima en sí aporta poca señal adicional.

**5. Las pruebas ciclistas, señal real pero puntual**
Las semanas con pruebas ciclistas generan picos de demanda claros, pero son eventos puntuales (pocas semanas al año) con importancia variable entre iteraciones.

### La variable que más querríamos tener y no tenemos

El principal factor ausente es el **stock disponible**. Si un producto está sin stock, las ventas son cero — no porque no haya demanda, sino porque no hay producto. El modelo interpreta esos ceros como "no hubo demanda" cuando en realidad puede ser "no hubo oferta". Incorporar información de stock podría ser el mayor salto que queda por dar.

---

## PARTE 4 — Ventajas e Inconvenientes de Cada Iteración

Esta tabla resume honestamente qué ganamos y qué perdemos o limitamos en cada decisión de diseño.

| Iteración | ✓ Ventajas | ✗ Inconvenientes / Limitaciones |
|---|---|---|
| **It1** Baseline | Sencillo, rápido, interpretable | Sin memoria, sin estacionalidad, sin ajuste |
| **It2** Memoria reciente | Captura tendencia reciente | Solo mira 4 semanas atrás; si hay anomalía reciente, el modelo la hereda |
| **It3** Estacionalidad | Captura ciclos anuales | Si el año fue atípico (ej. COVID), contamina la señal |
| **It4** Escala log | Gran mejora en error; equilibra productos grandes y pequeños | Más difícil de interpretar directamente (hay que revertir el logaritmo) |
| **It5** Historial SKU | La variable más potente del proyecto entra aquí | Productos nuevos (sin historial) no se benefician; riesgo de sobreajuste si hay pocos datos |
| **It6** Descuentos | Captura efecto promotor real (×4.2 de demanda) | Solo 1.5% de las filas tienen descuento — impacto global limitado |
| **It7** Dos modelos | Especialización, menos overfitting, MAPE muy mejorado | Mayor complejidad: dos modelos que mantener y monitorizar; nuevos SKUs deben clasificarse como A o B&C antes de predecir |

### El dilema de la complejidad

Cada mejora añade complejidad al sistema. Un modelo sencillo (It1) es fácil de mantener y de explicar a un equipo de negocio, pero predice peor. Un sistema de dos modelos especializados (It7) predice mejor, pero requiere más infraestructura para ponerlo en producción.

**La pregunta clave para la dirección:** ¿cuánto vale para CRUZBER mejorar el error de predicción un 20%? Si cada punto porcentual de mejora en la previsión de compra evita X euros de sobrestock o X euros de rotura de ventas, el coste de mantener dos modelos se justifica fácilmente.

---

## PARTE 5 — Evaluación Global del Modelo Actual (Iteración 7)

### ¿Es bueno el modelo? Una respuesta honesta.

**Sí, es útil y predice mejor que cualquier método simple.**
El modelo supera con claridad a cualquier alternativa básica como "vende lo mismo que la semana pasada" o "vende lo mismo que hace un año". Ha reducido el error un 20.8% respecto al punto de partida y predice con un error porcentual medio del 21.5%.

**Pero tiene limitaciones estructurales que hay que conocer.**
Un MAPE del 21.5% significa que en media nos equivocamos en más de 1 de cada 5 unidades. Para una empresa que necesita tomar decisiones de compra, esto es funcional pero no excelente. Las empresas líderes en forecasting (Amazon, Inditex) operan con errores por debajo del 10% — aunque con volúmenes y recursos de datos muy distintos.

### Dónde funciona bien

| Segmento / Zona | MAE | MAPE | Valoración |
|---|---|---|---|
| **Tipo B&C — Noroeste** | 0.314 | 15.4% | Excelente |
| **Tipo B&C — Noreste** | 0.362 | 15.8% | Muy bueno |
| **Tipo A — Noroeste** | 0.433 | 16.5% | Bueno |
| **Tipo A — Noreste** | 0.459 | 18.5% | Bueno |
| **Tipo B&C — Sur** | 0.392 | 16.7% | Bueno |

Las regiones con alta densidad de datos (Noreste con Barcelona/Valencia, Noroeste con Galicia) son donde el modelo funciona mejor. Más historia = mejor predicción.

### Dónde tiene dificultades

| Segmento / Zona | MAE | MAPE | Causa probable |
|---|---|---|---|
| **Tipo A — Norte** | 1.262 | 30.5% | Alta demanda + alta variabilidad en P. Vasco/Navarra |
| **Tipo A — Canarias** | 0.806 | 33.5% | Mercado insular con estacionalidad turística distinta |
| **Tipo A — Centro** | 0.902 | 25.1% | Alta demanda media (Madrid); picos difíciles de capturar |

El **Norte** (País Vasco, Navarra, Aragón) es la región más problemática para los productos estrella. La combinación de alta demanda y alta variabilidad produce los errores más grandes del modelo. Sería el primer candidato a un modelo específico en la siguiente fase.

### El overfitting: ¿el modelo aprende o memoriza?

Un buen modelo debe funcionar bien tanto en datos que conoce (train) como en datos nuevos (test). Si funciona muy bien en train pero mal en test, está "memorizando" en lugar de "aprender".

| Modelo | Error en datos conocidos | Error en datos nuevos | Brecha |
|---|---|---|---|
| Modelo A | MAE 0.572 | MAE 0.676 | +18.3% |
| Modelo B&C | MAE 0.367 | MAE 0.521 | +41.8% |

La brecha del modelo A es aceptable (18.3%). La del modelo B&C es alta (41.8%), pero tiene una explicación: los productos B&C tienen demanda tan esporádica que hay semanas con ventas que no tienen equivalente en el histórico — el modelo simplemente no las puede anticipar.

**Esta brecha no se resolverá con más algoritmos o más hiperparámetros. Se necesita nueva información** (stock, campañas planificadas, pedidos confirmados) que hoy no está disponible en el dataset.

### Referencia de contexto: ¿qué MAPE es "normal"?

| Sector / Empresa | MAPE típico | Contexto |
|---|---|---|
| Grandes retailers (Amazon, Inditex) | 5–12% | Miles de datos, información de stock y promociones en tiempo real |
| Retail mediano con forecasting maduro | 12–18% | Histórico largo, sistemas ERP integrados |
| **CRUZBER It7 (estado actual)** | **21.5%** | 3 años de datos, sin stock, sin calendario promo completo |
| Empresas sin modelo (intuición comprador) | 35–50% | Decisión humana sin datos estructurados |

El modelo ya supera con claridad a la previsión manual. El margen de mejora técnica hacia el 15% existe, pero requiere datos adicionales más que mejoras algorítmicas.

---

## PARTE 6 — Conclusión Final: Situación Actual y Próximos Pasos

### ¿Dónde estamos?

En 7 iteraciones de trabajo sistemático hemos construido un sistema de predicción de demanda que:

- Predice con un **error del 21.5%** (MAPE) la demanda semanal de cada SKU en cada municipio
- Ha reducido el error un **20.8% respecto al punto de partida** (de MAE=0.793 a MAE=0.628)
- Funciona mejor en productos B&C que en productos A a nivel de error porcentual
- Identifica correctamente los patrones estacionales, el efecto de los descuentos y el comportamiento histórico de cada producto en cada zona
- Opera con **dos modelos especializados** (Tipo A y Tipo B&C) que aprenden las reglas propias de cada tipo de producto

El modelo está en un estado **funcional y útil para el negocio**, aunque no ha llegado a su límite de mejora.

### ¿Por qué no hemos llegado más lejos?

El modelo actual tiene un techo impuesto por los datos disponibles, no por los algoritmos. Las principales fuentes de error que no podemos resolver con los datos actuales son:

1. **Roturas de stock**: cuando no hay producto, las ventas son cero, pero no porque no haya demanda. El modelo aprende esos ceros como demanda real y subestima en esas semanas.

2. **Calendario de campañas**: sabemos que los descuentos multiplican la demanda por 4.2, pero no tenemos el calendario de cuándo habrá descuentos en el futuro. El modelo lo descubre después, no antes.

3. **Factores externos no capturados**: competencia, tendencias de mercado, cambios en la gama de producto, variaciones de precio no registradas.

4. **Historial corto**: solo 3 años de datos (2022-2024) es poco para capturar ciclos largos o comportamientos excepcionales.

### Hoja de ruta: los próximos pasos por prioridad

**Prioridad 1 — Mejoras de modelo (sin datos nuevos)**

| Acción | Qué se busca | Impacto esperado |
|---|---|---|
| Modelo específico para Norte (Tipo A) | Reducir MAPE 30.5% en la región más problemática | Alto |
| Festivos autonómicos por comunidad | Capturar picos locales de demanda | Medio |
| Benchmark con LightGBM | Validar que CatBoost es la mejor arquitectura | Medio |

**Prioridad 2 — Datos nuevos (mayor impacto a largo plazo)**

| Dato a incorporar | Qué aportaría | Impacto esperado |
|---|---|---|
| **Stock disponible por semana** | Distinguir "no vendí porque no quise" de "no vendí porque no tenía" | Muy alto |
| **Calendario de promociones** | Predecir el efecto de los descuentos antes de que ocurran | Alto |
| **Festivos autonómicos** | Capturar el comportamiento local en puentes y fiestas regionales | Medio |
| **Datos de sell-out de competencia** | Entender si el mercado crece o CRUZBER pierde cuota | Alto |

**Prioridad 3 — Producción y uso real**

El mayor valor del modelo no está en el notebook sino en que el equipo comercial y de compras lo use. Para eso se necesita:

| Acción | Descripción |
|---|---|
| Pipeline de actualización automática | El modelo se reentrena cada semana con los datos nuevos |
| Interfaz para el equipo de compras | Dashboard donde ver las predicciones por zona y SKU |
| Sistema de alertas | Avisar cuando la predicción difiere mucho de lo esperado |
| Monitorización del rendimiento | Seguir el MAPE semana a semana para detectar degradaciones |

### El mensaje final

> El modelo es una herramienta de **apoyo a la decisión**, no un oráculo. Su valor está en reducir la incertidumbre del comprador, no en eliminarla. Una predicción con 21.5% de error es mucho mejor que ninguna predicción, y significativamente mejor que la estimación manual que se haría sin datos.
>
> La pregunta de negocio correcta no es "¿predice perfecto?" sino "¿predice mejor que lo que hacemos ahora, y el coste de implementarlo vale la mejora?". La respuesta a ambas preguntas es sí.
>
> **El siguiente paso más impactante es llevar el modelo a producción y conectarlo al proceso de compra de CRUZBER.**

---

## PARTE 7 — Cómo Presentar Estos Resultados al Comité de Dirección

### Principios para la presentación

Un comité de dirección no necesita entender cómo funciona CatBoost. Necesita entender **qué ganamos, cuánto cuesta y qué hacemos ahora**. La presentación debe responder tres preguntas en este orden:
1. ¿Qué problema tenemos?
2. ¿Qué hemos construido para resolverlo?
3. ¿Vale la pena y qué decidimos ahora?

---

### Estructura recomendada: 8 diapositivas

---

**DIAPOSITIVA 1 — El problema (1 min)**

*Título:* "Hoy compramos con incertidumbre. Podemos mejorar."

*Contenido:*
- CRUZBER gestiona X referencias en Y municipios
- Cada semana el equipo de compras toma decisiones sin una previsión de demanda fiable
- El error en estas decisiones tiene dos costes: **sobrestock** (capital inmovilizado) y **rotura** (ventas perdidas)
- Pregunta retórica: ¿cuánto cuesta equivocarse en la compra?

*Clave:* no hablar de modelos todavía. Hablar del dolor de negocio.

---

**DIAPOSITIVA 2 — La solución en una frase (30 seg)**

*Título:* "Hemos construido un sistema que predice cuánto se va a vender cada producto en cada zona la semana siguiente."

*Contenido (visual, sin texto denso):*
- Una flecha: DATOS HISTÓRICOS → MODELO → PREDICCIÓN SEMANAL
- Datos que usa el modelo: historial de ventas, precio, temporada, descuentos, clima, pruebas ciclistas
- Output: número de unidades previsto por SKU × municipio × semana

*Clave:* que entiendan el output, no el proceso.

---

**DIAPOSITIVA 3 — El viaje: 7 pasos para llegar aquí (2 min)**

*Título:* "7 iteraciones de mejora continua"

*Visual:* línea de tiempo horizontal con los 7 hitos y el MAE de cada uno.

| | It1 | It2 | It3 | It4 | It5 | It6 | It7 |
|---|---|---|---|---|---|---|---|
| **Error medio** | 0.793 | 0.773 | 0.769 | 0.649 | 0.641 | 0.641 | **0.628** |
| **Mejora acumulada** | — | −2.5% | −3% | −18% | −19% | −19% | **−21%** |

*Mensaje:* "Cada iteración es una hipótesis, un experimento y un resultado. Así funciona la ciencia de datos aplicada al negocio."

*Destacar:* el salto de It3 a It4 (transformación logarítmica) y el salto de It6 a It7 (dos modelos).

---

**DIAPOSITIVA 4 — El resultado en lenguaje de negocio (1 min)**

*Título:* "El modelo se equivoca de media en el 21.5% de la demanda"

*Contenido:*
- Si el modelo prevé que venderás 10 unidades, la realidad está entre 8 y 12 en el 78% de los casos
- Partíamos de un error del 26% (sin modelo dedicado)
- Las empresas líderes (Amazon, Inditex) operan al 5-12% con años de inversión y datos de stock en tiempo real
- **Sin modelo (estimación manual del comprador): se estima un error del 35-50%**

*Visual recomendado:* barra horizontal comparativa.
```
Manual del comprador:   ████████████████████████████████████  35-50%
Modelo inicial (It1):   ████████████████████████  ~26%
Modelo actual (It7):    ██████████████  21.5%
Objetivo futuro:        █████████  ~15%
```

*Mensaje:* "Ya somos significativamente mejores que la estimación manual. Hay recorrido hasta el 15% con datos adicionales."

---

**DIAPOSITIVA 5 — Lo que aprendimos sobre el negocio (2 min)**

*Título:* "El modelo nos enseña cosas que no sabíamos"

Tres hallazgos de negocio, no de modelo:

1. **Los descuentos promocionales multiplican la demanda por 4.2x**
   → Coordinar las previsiones de compra con el calendario de campañas es crítico

2. **El historial de ventas de cada producto en cada zona lo explica todo**
   → La mejor predicción para Bilbao semana que viene es lo que pasó en Bilbao históricamente, no la media nacional

3. **Los productos A y B&C tienen reglas de compra distintas**
   → Tratar igual a un best-seller y a un accesorio de nicho genera errores en ambos. Para los B&C, el precio es el driver; para los A, el historial y los descuentos

*Clave:* estos son insights accionables ahora, independientemente de si se pone el modelo en producción.

---

**DIAPOSITIVA 6 — Dónde funciona bien y dónde no (1 min)**

*Título:* "El modelo no predice igual de bien en todas las zonas"

*Visual:* mapa de España con colores (verde = bien, naranja = aceptable, rojo = mejorable)

| Zona | Error (MAPE) | Acción recomendada |
|---|---|---|
| Noroeste / Noreste | 15-19% | Listo para usar en decisiones de compra |
| Sur / Centro | 22-25% | Usar con margen de seguridad |
| Norte / Canarias | 30-35% | Requiere modelo específico o revisión manual |

*Mensaje:* "Podemos empezar a usarlo en las zonas donde funciona bien mientras mejoramos las que peor se comportan."

---

**DIAPOSITIVA 7 — El plan y lo que necesitamos (2 min)**

*Título:* "Tres pasos para ir del laboratorio a la toma de decisiones"

**Paso 1 — Corto plazo (próximas semanas):** mejoras técnicas sin datos nuevos
- Modelo específico para Norte
- Benchmark con algoritmo alternativo
- Festivos autonómicos

**Paso 2 — Medio plazo (1-3 meses):** datos nuevos de alto impacto
- Conectar información de stock disponible
- Incorporar calendario de campañas y descuentos planificados
- Meta: bajar del 18% de MAPE

**Paso 3 — Producción (3-6 meses):** llevar el modelo al proceso de compra
- Dashboard de predicciones semanal para el equipo de compras
- Actualización automática con datos nuevos
- Sistema de monitorización del error

*Inversión estimada:* [a completar por el equipo técnico con estimación de horas/recursos]

---

**DIAPOSITIVA 8 — La decisión (30 seg)**

*Título:* "¿Qué decidimos hoy?"

Tres opciones para el comité:

| Opción | Descripción | Inversión |
|---|---|---|
| **A — Continuar** | Seguir iterando el modelo con nuevas mejoras técnicas | Baja |
| **B — Pilotar** | Usar el modelo actual en una zona / categoría piloto | Media |
| **C — Escalar** | Plan completo hacia producción con nuevos datos | Alta |

*Recomendación del equipo:* Opción B — pilotar en Noroeste + Noreste (donde el modelo funciona mejor) para generar evidencia de impacto en negocio antes de escalar.

---

### Consejos para la presentación

- **Duración total**: 10-12 minutos + preguntas. No más.
- **No usar el word "algoritmo" ni "hiperparámetro"**. Sustituir por "el motor del modelo" o "la configuración del modelo".
- **Llevar un ejemplo concreto**: "Esta semana el modelo predijo X unidades de [referencia] en Bilbao. La realidad fue Y. Eso es un error del Z%." Lo concreto convence más que los promedios.
- **Anticipar la pregunta del ROI**: cuantificar cuánto costaría equivocarse en la compra de los productos más vendidos. Si ese número es grande, la inversión en el modelo se justifica sola.
- **No prometer perfección**: el 21.5% de error hay que presentarlo como un logro respecto al punto de partida, no como una limitación. "Pasamos del 35-50% de error manual al 21.5%. El modelo funciona."